# In This project I will build my own LLM from scratch . I want to build a cs_ tutor LLM.

# Mini GPT From Scratch — TinyStories

## Introduction

In this project, I implemented and trained a **GPT-style language model from scratch** using the **TinyStories dataset**. The goal of this project was to understand the complete pipeline behind modern large language models, including tokenization, dataset preparation, transformer architecture, training, checkpointing, and text generation.

Instead of relying on a pretrained model, this project builds the entire system using **PyTorch** and trains it directly on text data. This allows a deeper understanding of how transformer-based language models learn to predict the next token in a sequence.

The workflow of this project includes:

* Loading and preparing the TinyStories dataset
* Training and loading a tokenizer
* Building a GPT-style transformer architecture
* Training the model with validation monitoring
* Saving checkpoints during training
* Loading the best checkpoint
* Generating text from prompts

Through this project, I explored how language models learn patterns in text and the challenges involved when training models from scratch with limited compute resources.


In [1]:
import torch
print (torch.__version__)
print (torch.cuda.is_available())
print("GPU Namem: ", torch.cuda.get_device_name(0))

2.10.0+cu128
True
GPU Namem:  NVIDIA RTX PRO 6000 Blackwell Server Edition


# Data Setup

### Install required Library

In [2]:
# Install the library we need
#!pip install -q transformers datasets accelerate sentencepiece tokenizers

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# create project folder
import os
PROJECT_PATH  = "/content/drive/MyDrive/mini_llm_project"
os.makedirs( PROJECT_PATH, exist_ok = True)
print("Project folder ready : ", PROJECT_PATH)

Project folder ready :  /content/drive/MyDrive/mini_llm_project


### Load TiniStories Dataset

In [5]:
from datasets import load_dataset
dataset = load_dataset("eminorhan/tinystories", "10M_1")
print(dataset)

#

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 62345
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 27635
    })
})


### Check the data

In [6]:
print(dataset["train"][0])

{'text': 'Once upon a time, there was a little dog named Max. Max had a hurt leg. He could not run or play with his friends. Max was sad. He looked at the distant park, where his friends were playing.\nOne day, Max saw a big bird. The bird said, "Max, I can help you. I will include you in my flying fun." Max was happy. He wanted to fly with the bird.\nMax and the bird started to fly. They flew high in the sky. Max felt happy and free. But then, the bird let go of Max. Max fell to the ground. The bird said, "You should learn to be happy with what you have."\nMax learned that he should be happy with his leg. He knew he could not fly. Max was sad, but he knew the bird was right. The moral of the story is to be happy with what you have.'}


# Build the tokenizer

### Extract text from dataset

In [7]:
# collecting training text
def get_training_corpus():
  for i in range(0, len(dataset["train"]), 1000 ):
    yield dataset["train"][i: i+1000]["text"]

### Import tokenizer tools

In [8]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

### Create Empty Tokenizer

In [9]:
# Intialize tokenizer model
tokenizer = Tokenizer(models.BPE())

### Add pre_tokenizer

In [10]:
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

### Train the tokenizer

In [11]:
trainer = trainers.BpeTrainer(vocab_size = 16000,
                              special_tokens= [ "[PAD]", "[UNK]", "[BOS]", "[EOS]"])
tokenizer.train_from_iterator(get_training_corpus(), trainer = trainer)

### Test the Tokenizer

In [12]:
encoded = tokenizer.encode("Hello, I want to learn computer science.")
print(encoded.tokens)
print(encoded.ids)

['Hello', ',', 'I', 'want', 'to', 'learn', 'computer', 'science', '.']
[913, 12, 37, 331, 99, 391, 2767, 8599, 14]


In [13]:
print("Vocabulary size: ", tokenizer.get_vocab_size())

Vocabulary size:  16000


# Build a mini model from scratch

In [14]:
# import + Reproducibility
import math, os, random
import torch.nn as nn
import torch.nn.functional as F
# make result more repeatable
torch.manual_seed(42)
random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device: ", device)

Device:  cuda


### Define model settings

In [15]:
# ===== Model size (small but real) =====
vocab_size = tokenizer.get_vocab_size()   # should be ~16000
block_size = 128      # max tokens the model reads at once
n_embd = 256          # how big each token's "meaning vector" is
n_head = 8            # number of attention heads
n_layer = 4           # number of transformer blocks
dropout = 0.1

print("vocab_size:", vocab_size)

vocab_size: 16000


### Build attention

In [20]:
#  One Attention Head
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape  # Batch, Time, Channels(embedding)

        k = self.key(x)      # (B, T, head_size)
        q = self.query(x)    # (B, T, head_size)

        # attention scores: (B, T, T)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)

        # mask future tokens (so it can't cheat)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        # softmax -> probabilities
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)    # (B, T, head_size)
        out = wei @ v        # (B, T, head_size)
        return out

In [21]:
# Multi_head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [22]:
# FeedForward
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [23]:
# Transformer Block
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))   # attention + skip connection
        x = x + self.ffwd(self.ln2(x)) # feedforward + skip connection
        return x

In [24]:
# Mini Model
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)                         # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embd)
        x = tok_emb + pos_emb                                             # (B, T, n_embd)

        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                          # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))

        return logits, loss

In [25]:
# quick test
model = MiniGPT().to(device)
print("Model parameters:", sum(p.numel() for p in model.parameters())/1e6, "M")

# dummy batch (random tokens)
x = torch.randint(0, vocab_size, (2, 32)).to(device)
logits, loss = model(x, x)
print("Logits shape:", logits.shape)
print("Loss:", loss.item())

Model parameters: 11.397248 M
Logits shape: torch.Size([2, 32, 16000])
Loss: 9.866266250610352


### Tokenize dataset

In [26]:
# create a Tokeniztion Function
def tokenize_function(example):
    tokens = tokenizer.encode(example["text"]).ids
    return {"ids": tokens}

In [27]:
# apply it to whole dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=["text"]
)

Map:   0%|          | 0/62345 [00:00<?, ? examples/s]

Map:   0%|          | 0/27635 [00:00<?, ? examples/s]

### Merging all tokens

In [28]:
def group_texts(examples):
    # 1) Concatenate all token lists into one long list
    concatenated = sum(examples["ids"], [])

    # 2) We need at least block_size+1 so we can make (input, next_token_label)
    total_length = (len(concatenated) // (block_size + 1)) * (block_size + 1)
    concatenated = concatenated[:total_length]

    input_ids = []
    labels = []

    # 3) Make chunks of length block_size+1
    for i in range(0, total_length, block_size + 1):
        chunk = concatenated[i : i + block_size + 1]

        # inputs are first block_size tokens
        x = chunk[:-1]   # length = block_size

        # labels are next-token targets (shifted)
        y = chunk[1:]    # length = block_size

        input_ids.append(x)
        labels.append(y)

    return {"input_ids": input_ids, "labels": labels}

In [29]:
lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True,
    remove_columns=tokenized_dataset["train"].column_names,
)

Map:   0%|          | 0/62345 [00:00<?, ? examples/s]

Map:   0%|          | 0/27635 [00:00<?, ? examples/s]

### Convert to PyTorch Format

In [30]:
lm_dataset.set_format(type="torch")

In [31]:
batch = lm_dataset["train"][0]
print(len(batch["input_ids"]), len(batch["labels"]))
print(batch["input_ids"][:10])
print(batch["labels"][:10])

128 128
tensor([239, 244,  58, 201,  12, 211, 109,  58, 200, 271])
tensor([244,  58, 201,  12, 211, 109,  58, 200, 271, 213])


### create DataLoader

In [32]:
from torch.utils.data import DataLoader

batch_size = 16

train_loader = DataLoader(lm_dataset["train"], batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(lm_dataset["validation"], batch_size=batch_size, shuffle=False)

# quick check
batch = next(iter(train_loader))
print(batch["input_ids"].shape, batch["labels"].shape)

torch.Size([16, 128]) torch.Size([16, 128])


### Create Optimizer

In [33]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

### Training

In [34]:


# model.train()
# print_every = 200

# for step, batch in enumerate(train_loader):
#     input_ids = batch["input_ids"].to(device)
#     labels    = batch["labels"].to(device)

#     logits, loss = model(input_ids, labels)

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

#     if step % print_every == 0:
#         print(f"Step {step}, Loss: {loss.item():.4f}")

#     if step >= 2000:   # stop early for first test run
#         break

### Improving the model

In [35]:
# Add a validation function
@torch.no_grad()
def estimate_loss(model, train_loader, val_loader, eval_iters=50):
    model.eval()
    out = {}
    for split, loader in [("train", train_loader), ("val", val_loader)]:
        losses = []
        for i, batch in enumerate(loader):
            if i >= eval_iters:
                break
            x = batch["input_ids"].to(device)
            y = batch["labels"].to(device)
            _, loss = model(x, y)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

In [36]:
#Upgrade  optimizer + scheduler + clipping
from torch.optim.lr_scheduler import CosineAnnealingLR

lr = 3e-4
weight_decay = 0.1
max_steps = 10000          # stronger than 2000
eval_every = 500
save_every = 1000
grad_clip = 1.0

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=max_steps)

### Retrain

In [37]:
# import os, torch

# CKPT_DIR = os.path.join(PROJECT_PATH, "checkpoints")
# os.makedirs(CKPT_DIR, exist_ok=True)

# model.train()
# step = 0

# while step < max_steps:
#     for batch in train_loader:
#         x = batch["input_ids"].to(device)
#         y = batch["labels"].to(device)

#         _, loss = model(x, y)

#         optimizer.zero_grad()
#         loss.backward()

#         # stabilize training
#         torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

#         optimizer.step()
#         scheduler.step()

#         if step % 200 == 0:
#             print(f"step {step} | loss {loss.item():.4f} | lr {scheduler.get_last_lr()[0]:.2e}")

#         # validation check
#         if step % eval_every == 0 and step > 0:
#             losses = estimate_loss(model, train_loader, val_loader, eval_iters=50)
#             print(f"[eval] step {step} | train {losses['train']:.4f} | val {losses['val']:.4f}")

#         # save checkpoint
#         if step % save_every == 0 and step > 0:
#             ckpt_path = os.path.join(CKPT_DIR, f"minigpt_step{step}.pt")
#             torch.save({
#                 "step": step,
#                 "model_state": model.state_dict(),
#                 "optimizer_state": optimizer.state_dict(),
#                 "vocab_size": vocab_size,
#                 "block_size": block_size,
#                 "n_embd": n_embd,
#                 "n_head": n_head,
#                 "n_layer": n_layer
#             }, ckpt_path)
#             print("Saved:", ckpt_path)

#         step += 1
#         if step >= max_steps:
#             break

## Load the best check point

In [38]:
import torch, os

ckpt_path = os.path.join(PROJECT_PATH, "checkpoints", "minigpt_step9000.pt")

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.to(device)
model.eval()

print("Loaded checkpoint:", ckpt_path)
print("Step:", ckpt["step"])

Loaded checkpoint: /content/drive/MyDrive/mini_llm_project/checkpoints/minigpt_step9000.pt
Step: 9000


In [ ]:
# # Save a “final model” file
# final_path = os.path.join(PROJECT_PATH, "final_minigpt.pt")
# torch.save(model.state_dict(), final_path)
# print("Saved final weights:", final_path)

# # tokenizer already saved earlier as tokenizer.json
# print("Tokenizer file:", os.path.join(PROJECT_PATH, "tokenizer.json"))

### Generation part

In [41]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def generate(model, start_text, max_new_tokens=80, temperature=0.9, top_k=50):
    model.eval()

    ids = tokenizer.encode(start_text).ids
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        x_cond = x[:, -block_size:]  # keep last block_size tokens
        logits, _ = model(x_cond)

        logits = logits[:, -1, :]  # last position
        logits = logits / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist())

In [42]:
print(generate(model, "Once upon a time", max_new_tokens=120))

Once upon a time , there was a little girl named Lily . She had a big smile in her room . It was her favorite toy . The cat liked the toy and made a new room . One sunny day , Lily found a big , shiny doll . Lily did not know what her mom was in the toy box . The doll was very useful and shiny , but they were very pretty . Lily felt sad and she decided to have a new toy . She asked her mom if she gave the doll to clean up . She said yes , and it could play with the doll . Lily felt happy and safe . She took the


In [43]:
print(generate(model, "The little dog", max_new_tokens=120))
print(generate(model, "In a small village", max_new_tokens=120))

The little dog smiled and said , " I can help you ." The dog and the cat tried to find its home . They all worked together and now the big dog ' s house . They found the big tree and put it in the shade . The little cat and the little cat laughed as they played with the big dog . They played together all day and had a great day . Once upon a time , there was a little girl named Lily . Lily loved to play with her toy pistol in the park . She was the best of animals , always came to the park to play . Lily had an idea . She asked
In a small village , a girl named Lily found a big box . The box was very pretty . Lily had a toy box in her room . She picked it up and said , " Look , I found this toy is not yours . I ' m inside !" Once upon a time , there was a big cow . The cow looked and nice . The her pig lived in a small town . The cow could not touch the pig . One day , a little boy named Tim went for a walk . He saw a big rock . The rock was very pretty . Tim wanted to play , but he wa

## Evaluate the scratch model

In [44]:
@torch.no_grad()
def eval_scratch(model, loader, eval_iters=100):
    model.eval()
    losses = []

    for i, batch in enumerate(loader):
        if i >= eval_iters:
            break

        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)

        _, loss = model(x, y)
        losses.append(loss.item())

    return sum(losses) / len(losses)


scratch_val_loss = eval_scratch(model, val_loader)

print("Scratch Model Validation Loss:", scratch_val_loss)
print("Scratch Perplexity:", torch.exp(torch.tensor(scratch_val_loss)).item())

Scratch Model Validation Loss: 2.4442915534973144
Scratch Perplexity: 11.522383689880371


# STEP 2
we built our model function  from sratch now we will try the use the build fonction and compare them to see the difference

### Build the “built” model

In [16]:
# ===== Model size (small but real) =====
vocab_size = tokenizer.get_vocab_size()   # should be ~16000
block_size = 256     # max tokens the model reads at once
n_embd = 768        # how big each token's "meaning vector" is
n_head = 12           # number of attention heads
n_layer = 12           # number of transformer blocks
dropout = 0.1

print("vocab_size:", vocab_size)

vocab_size: 16000


In [17]:
from transformers import GPT2Config, GPT2LMHeadModel

built_config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=n_embd,
    n_layer=n_layer,
    n_head=n_head,
    resid_pdrop=dropout,
    embd_pdrop=dropout,
    attn_pdrop=dropout,
)

# enable weight tying
built_config.tie_word_embeddings = True
built_model = GPT2LMHeadModel(built_config).to(device)
built_model.train()

print("Built model params (M):", sum(p.numel() for p in built_model.parameters())/1e6)

Built model params (M): 97.540608


### Train the built model

In [40]:
# import time
# from torch.optim import AdamW

# built_optimizer = AdamW(built_model.parameters(), lr=3e-4, weight_decay=0.1)

# max_steps = 2000
# print_every = 200

# t0 = time.time()

# for step, batch in enumerate(train_loader):
#     if step >= max_steps:
#         break

#     x = batch["input_ids"].to(device)
#     y = batch["labels"].to(device)

#     out = built_model(input_ids=x, labels=y)
#     loss = out.loss

#     built_optimizer.zero_grad()
#     loss.backward()
#     torch.nn.utils.clip_grad_norm_(built_model.parameters(), 1.0)
#     built_optimizer.step()

#     if step % print_every == 0:
#         steps_per_sec = (step + 1) / (time.time() - t0)
#         print(f"[built] step {step} | loss {loss.item():.4f} | {steps_per_sec:.2f} steps/s")

### Evaluate built model

In [ ]:
# @torch.no_grad()
# def eval_built(model, loader, eval_iters=100):
#     model.eval()
#     losses = []

#     for i, batch in enumerate(loader):
#         if i >= eval_iters:
#             break

#         x = batch["input_ids"].to(device)
#         y = batch["labels"].to(device)

#         out = model(input_ids=x, labels=y)
#         losses.append(out.loss.item())

#     return sum(losses) / len(losses)

# built_val_loss = eval_built(built_model, val_loader)

# print("Built Model Validation Loss:", built_val_loss)
# print("Built Perplexity:", torch.exp(torch.tensor(built_val_loss)).item())

Built Model Validation Loss: 3.800534908771515
Built Perplexity: 44.72510528564453


In [ ]:
# # continue training from where you stopped
# import time
# from torch.optim import AdamW

# # only create optimizer if you didn't already
# # if you already have built_optimizer, skip this line
# built_optimizer = AdamW(built_model.parameters(), lr=3e-4, weight_decay=0.1)

# max_steps = 10000
# print_every = 200

# t0 = time.time()

# for step, batch in enumerate(train_loader):
#     # IMPORTANT: step restarts at 0 in enumerate, so we just run extra steps.
#     # If you want exact 10k total steps, set this to 8000 more steps instead.
#     if step >= 8000:   # 2000 already done, do 8000 more
#         break

#     x = batch["input_ids"].to(device)
#     y = batch["labels"].to(device)

#     out = built_model(input_ids=x, labels=y)
#     loss = out.loss

#     built_optimizer.zero_grad()
#     loss.backward()
#     torch.nn.utils.clip_grad_norm_(built_model.parameters(), 1.0)
#     built_optimizer.step()

#     if step % print_every == 0:
#         steps_per_sec = (step + 1) / (time.time() - t0)
#         print(f"[built-continue] +{step} steps | loss {loss.item():.4f} | {steps_per_sec:.2f} steps/s")

[built-continue] +0 steps | loss 3.8286 | 33.62 steps/s
[built-continue] +200 steps | loss 4.0052 | 83.74 steps/s
[built-continue] +400 steps | loss 3.7319 | 84.19 steps/s
[built-continue] +600 steps | loss 3.6456 | 84.59 steps/s
[built-continue] +800 steps | loss 3.6710 | 84.88 steps/s
[built-continue] +1000 steps | loss 3.7174 | 85.22 steps/s
[built-continue] +1200 steps | loss 3.6604 | 85.39 steps/s
[built-continue] +1400 steps | loss 3.5959 | 85.43 steps/s
[built-continue] +1600 steps | loss 3.5259 | 85.47 steps/s
[built-continue] +1800 steps | loss 3.5524 | 85.23 steps/s
[built-continue] +2000 steps | loss 3.3673 | 85.14 steps/s
[built-continue] +2200 steps | loss 3.5386 | 85.13 steps/s
[built-continue] +2400 steps | loss 3.4499 | 84.49 steps/s
[built-continue] +2600 steps | loss 3.4454 | 84.09 steps/s
[built-continue] +2800 steps | loss 3.3334 | 84.08 steps/s
[built-continue] +3000 steps | loss 3.1647 | 84.28 steps/s
[built-continue] +3200 steps | loss 3.3263 | 84.44 steps/s
[bui

In [39]:
# built_val_loss = eval_built(built_model, val_loader)
# print("Built Model Validation Loss:", built_val_loss)
# print("Built Perplexity:", torch.exp(torch.tensor(built_val_loss)).item())

### Note
my scratch method beat the build method but this was possible because I train my scratch one very good than the build one .
For the net part I will continue with the build one to make my life easier

### Load TinyStories 100M_1



In [18]:
from datasets import load_dataset

dataset_100m = load_dataset("eminorhan/tinystories", "100M_1")
print(dataset_100m)
print(dataset_100m["train"][0]["text"][:200])

100M_1/train-00000-of-00002.parquet:   0%|          | 0.00/130M [00:00<?, ?B/s]

100M_1/train-00001-of-00002.parquet:   0%|          | 0.00/130M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/622827 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/27635 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 622827
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 27635
    })
})
Once upon a time, there was a little boy named Tim. Tim had a toy bear named Ben. Tim and Ben liked to play all day. They would run, jump, and take big steps together. They were best friends.
One day,


### Tokenize the 100M dataset

In [19]:
EOS_ID = tokenizer.token_to_id("[EOS]")
assert EOS_ID is not None, "Tokenizer does not have [EOS] token"

def tokenize_with_eos(examples):
    ids_list = []
    for text in examples["text"]:
        ids = tokenizer.encode(text).ids
        ids.append(EOS_ID)          # end-of-story
        ids_list.append(ids)
    return {"ids": ids_list}

tok_100m = dataset_100m.map(
    tokenize_with_eos,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/622827 [00:00<?, ? examples/s]

Map:   0%|          | 0/27635 [00:00<?, ? examples/s]

In [20]:
# def tokenize_batch(batch):
#     # batch["text"] is a list of strings
#     enc = [tokenizer.encode(t).ids for t in batch["text"]]
#     return {"input_ids": enc}

# tokenized_100m = dataset_100m.map(
#     tokenize_batch,
#     batched=True,
#     remove_columns=["text"],
# )
# print(tokenized_100m)

In [21]:
block_size = 256   # keep your choice here

def make_blocks(examples):
    input_ids = []
    labels = []

    for ids in examples["ids"]:
        # slide through ONE story at a time
        for i in range(0, len(ids) - 1, block_size):
            chunk = ids[i : i + block_size + 1]

            # need at least 2 tokens to make x/y
            if len(chunk) < 2:
                continue

            x = chunk[:-1]
            y = chunk[1:]

            input_ids.append(x)
            labels.append(y)

    return {"input_ids": input_ids, "labels": labels}

lm_100m = tok_100m.map(
    make_blocks,
    batched=True,
    remove_columns=["ids"]
)

Map:   0%|          | 0/622827 [00:00<?, ? examples/s]

Map:   0%|          | 0/27635 [00:00<?, ? examples/s]

### Group tokens into fixed blocks

### DataLoader

In [22]:
from torch.utils.data import DataLoader

batch_size = 8

train_loader_100m = DataLoader(lm_100m["train"], batch_size=batch_size, shuffle=True)
val_loader_100m = DataLoader(lm_100m["validation"], batch_size=batch_size, shuffle=False)

### build the model for 100M

### Evaluation

In [23]:
from transformers import GPT2Config, GPT2LMHeadModel

built_config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=n_embd,
    n_layer=n_layer,
    n_head=n_head,
    resid_pdrop=dropout,
    embd_pdrop=dropout,
    attn_pdrop=dropout,
)

# enable weight tying
built_config.tie_word_embeddings = True
built_model_100m = GPT2LMHeadModel(built_config).to(device)
built_model_100m.train()

print("Built model params (M):", sum(p.numel() for p in built_model.parameters())/1e6)

Built model params (M): 97.540608


In [24]:
import torch, math
from contextlib import nullcontext

@torch.no_grad()
def eval_built(model, loader, eval_iters=100):
    model.eval()
    losses = []
    for i, batch in enumerate(loader):
        if i >= eval_iters:
            break
        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)
        out = model(input_ids=x, labels=y)
        losses.append(out.loss.item())
    model.train()
    avg = sum(losses) / len(losses)
    ppl = math.exp(avg)
    return avg, ppl

### Training

In [57]:
# import os, time
# import torch
# from torch.optim import AdamW

# PROJECT_PATH = "/content/drive/MyDrive/mini_llm_project"
# CKPT_DIR = os.path.join(PROJECT_PATH, "checkpoints_built_100m")
# os.makedirs(CKPT_DIR, exist_ok=True)

# model = built_model_100m  # use the model you created
# model.train()

# lr = 3e-4
# weight_decay = 0.1
# max_steps = 10000          # start with 10k, we can extend later
# print_every = 200
# eval_every = 1000
# eval_iters = 100

# grad_accum_steps = 4       # effective batch = batch_size * 4
# use_amp = True

# optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# best_val = float("inf")
# best_path = None

# t0 = time.time()
# step = 0
# optimizer.zero_grad()

# train_iter = iter(train_loader_100m)

# while step < max_steps:
#     # get batch (loop forever)
#     try:
#         batch = next(train_iter)
#     except StopIteration:
#         train_iter = iter(train_loader_100m)
#         batch = next(train_iter)

#     x = batch["input_ids"].to(device)
#     y = batch["labels"].to(device)

#     autocast_ctx = torch.cuda.amp.autocast(enabled=use_amp)
#     with autocast_ctx:
#         out = model(input_ids=x, labels=y)
#         loss = out.loss / grad_accum_steps  # IMPORTANT

#     scaler.scale(loss).backward()

#     # update weights every grad_accum_steps
#     if (step + 1) % grad_accum_steps == 0:
#         scaler.unscale_(optimizer)
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         scaler.step(optimizer)
#         scaler.update()
#         optimizer.zero_grad()

#     # logging
#     if step % print_every == 0:
#         steps_per_sec = (step + 1) / (time.time() - t0)
#         print(f"[100M built] step {step} | train_loss {loss.item()*grad_accum_steps:.4f} | {steps_per_sec:.2f} steps/s")

#     # eval + save best
#     if step % eval_every == 0 and step > 0:
#         val_loss, val_ppl = eval_built(model, val_loader_100m, eval_iters=eval_iters)
#         print(f"[EVAL] step {step} | val_loss {val_loss:.4f} | val_ppl {val_ppl:.2f}")

#         ckpt_path = os.path.join(CKPT_DIR, f"built100m_step{step}.pt")
#         torch.save({
#             "step": step,
#             "model_state": model.state_dict(),
#             "optimizer_state": optimizer.state_dict(),
#             "best_val": best_val,
#             "vocab_size": vocab_size,
#             "block_size": block_size,
#             "n_embd": n_embd,
#             "n_head": n_head,
#             "n_layer": n_layer,
#         }, ckpt_path)
#         print("Saved:", ckpt_path)

#         if val_loss < best_val:
#             best_val = val_loss
#             best_path = os.path.join(CKPT_DIR, "BEST.pt")
#             torch.save({
#                 "step": step,
#                 "model_state": model.state_dict(),
#                 "optimizer_state": optimizer.state_dict(),
#                 "best_val": best_val,
#             }, best_path)
#             print("✅ New BEST saved:", best_path)

#     step += 1

# print("Done. Best val loss:", best_val)
# print("Best checkpoint:", best_path)

/tmp/ipykernel_3202/3253929329.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_3202/3253929329.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  autocast_ctx = torch.cuda.amp.autocast(enabled=use_amp)


[100M built] step 0 | train_loss 5.0670 | 46.47 steps/s
[100M built] step 200 | train_loss 4.5609 | 53.23 steps/s
[100M built] step 400 | train_loss 4.7015 | 53.54 steps/s
[100M built] step 600 | train_loss 4.3239 | 53.64 steps/s
[100M built] step 800 | train_loss 4.3248 | 53.70 steps/s
[100M built] step 1000 | train_loss 3.9994 | 53.73 steps/s
[EVAL] step 1000 | val_loss 4.0996 | val_ppl 60.32
Saved: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/built100m_step1000.pt
✅ New BEST saved: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt
[100M built] step 1200 | train_loss 4.1234 | 24.04 steps/s
[100M built] step 1400 | train_loss 4.1686 | 26.10 steps/s
[100M built] step 1600 | train_loss 4.1139 | 27.90 steps/s
[100M built] step 1800 | train_loss 3.6468 | 29.48 steps/s
[100M built] step 2000 | train_loss 3.9612 | 30.87 steps/s
[EVAL] step 2000 | val_loss 3.7379 | val_ppl 42.01
Saved: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/built

In [26]:
import torch, os

best_path = "/content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt"

ckpt = torch.load(best_path, map_location=device)
built_model_100m.load_state_dict(ckpt["model_state"])
built_model_100m.to(device)
built_model_100m.eval()

print("Loaded BEST checkpoint at step:", ckpt["step"])
print("Best val loss:", ckpt["best_val"])

Loaded BEST checkpoint at step: 48000
Best val loss: 2.3162708830833436


In [29]:
lens = [len(lm_100m["train"][i]["input_ids"]) for i in range(20)]
print(lens)

[181, 173, 175, 165, 188, 192, 219, 168, 256, 60, 168, 166, 256, 141, 256, 5, 154, 193, 146, 137]


In [30]:
import torch

PAD_ID = tokenizer.token_to_id("[PAD]")
if PAD_ID is None:
    PAD_ID = 0  # fallback

def collate_pad(batch):
    input_ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
    labels    = [torch.tensor(x["labels"], dtype=torch.long) for x in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=PAD_ID)
    labels    = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)  # ignore padding in loss

    return {"input_ids": input_ids, "labels": labels}

In [31]:
from torch.utils.data import DataLoader

batch_size = 8  # or 8 if GPU allows
train_loader_100m = DataLoader(lm_100m["train"], batch_size=batch_size, shuffle=True, collate_fn=collate_pad)
val_loader_100m   = DataLoader(lm_100m["validation"], batch_size=batch_size, shuffle=False, collate_fn=collate_pad)

In [32]:
import time, math
import torch
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

PROJECT_PATH = "/content/drive/MyDrive/mini_llm_project"
CKPT_DIR = os.path.join(PROJECT_PATH, "checkpoints_built_100m")
os.makedirs(CKPT_DIR, exist_ok=True)

model = built_model_100m

# ====== SETTINGS (LLM style) ======
target_steps = 50000            # train longer
print_every = 200
eval_every  = 2000
eval_iters  = 100

lr = 3e-4
weight_decay = 0.1
grad_accum_steps = 4
max_grad_norm = 1.0
use_amp = True

# ====== OPTIMIZER ======
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# If BEST checkpoint stored optimizer state, load it (so resume is real)
if "optimizer_state" in ckpt:
    optimizer.load_state_dict(ckpt["optimizer_state"])
    print("✅ Optimizer state loaded")

# ====== SCHEDULER (Warmup + Cosine) ======
# Scheduler steps happen only when optimizer.step() happens
# total optimizer updates = total_steps / grad_accum_steps
start_step = int(ckpt.get("step", 0))
total_updates = target_steps // grad_accum_steps
done_updates  = start_step // grad_accum_steps

warmup_updates = int(0.05 * total_updates)  # 5% warmup

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_updates,
    num_training_steps=total_updates,
)

# Fast-forward scheduler if resuming
for _ in range(done_updates):
    scheduler.step()

print("Scheduler ready:",
      "total_updates =", total_updates,
      "warmup_updates =", warmup_updates,
      "done_updates =", done_updates)

# ====== MODERN AMP ======
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_val = float(ckpt.get("best_val", float("inf")))
best_path = os.path.join(CKPT_DIR, "BEST.pt")

# ====== TRAIN LOOP ======
t0 = time.time()
step = start_step
optimizer.zero_grad(set_to_none=True)

train_iter = iter(train_loader_100m)

while step < target_steps:
    # get batch (loop forever)
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader_100m)
        batch = next(train_iter)

    x = batch["input_ids"].to(device)
    y = batch["labels"].to(device)

    with torch.amp.autocast("cuda", enabled=use_amp):
        out = model(input_ids=x, labels=y)
        loss = out.loss / grad_accum_steps

    scaler.scale(loss).backward()

    # update every grad_accum_steps
    if (step + 1) % grad_accum_steps == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        scheduler.step()  # <-- THIS is the new big improvement!

    if step % print_every == 0:
        lr_now = scheduler.get_last_lr()[0]
        steps_per_sec = (step - start_step + 1) / (time.time() - t0)
        print(f"[100M built] step {step} | train_loss {(loss.item()*grad_accum_steps):.4f} | lr {lr_now:.2e} | {steps_per_sec:.2f} steps/s")

    # ====== EVAL + SAVE ======
    if step % eval_every == 0 and step > start_step:
        val_loss, val_ppl = eval_built(model, val_loader_100m, eval_iters=eval_iters)
        print(f"[EVAL] step {step} | val_loss {val_loss:.4f} | val_ppl {val_ppl:.2f}")

        # Save step checkpoint
        ckpt_path = os.path.join(CKPT_DIR, f"built100m_step{step}.pt")
        torch.save({
            "step": step,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val": best_val,
        }, ckpt_path)
        print("Saved:", ckpt_path)

        # Save BEST
        if val_loss < best_val:
            best_val = val_loss
            torch.save({
                "step": step,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val": best_val,
            }, best_path)
            print("✅ New BEST saved:", best_path)

    step += 1

print("Done. Best val loss:", best_val)
print("Best checkpoint:", best_path)

✅ Optimizer state loaded
Scheduler ready: total_updates = 12500 warmup_updates = 625 done_updates = 12000


/tmp/ipykernel_3142/774409501.py:49: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


[100M built] step 48000 | train_loss 2.4081 | lr 1.31e-06 | 1.54 steps/s
[100M built] step 48200 | train_loss 2.2470 | lr 1.06e-06 | 44.39 steps/s
[100M built] step 48400 | train_loss 2.1877 | lr 8.39e-07 | 49.07 steps/s
[100M built] step 48600 | train_loss 2.4729 | lr 6.43e-07 | 51.05 steps/s
[100M built] step 48800 | train_loss 2.4395 | lr 4.72e-07 | 52.23 steps/s
[100M built] step 49000 | train_loss 2.1867 | lr 3.28e-07 | 52.95 steps/s
[100M built] step 49200 | train_loss 2.0031 | lr 2.10e-07 | 53.34 steps/s
[100M built] step 49400 | train_loss 2.4886 | lr 1.18e-07 | 53.60 steps/s
[100M built] step 49600 | train_loss 2.4324 | lr 5.25e-08 | 53.85 steps/s
[100M built] step 49800 | train_loss 2.0850 | lr 1.31e-08 | 54.06 steps/s
Done. Best val loss: 2.3162708830833436
Best checkpoint: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt


In [33]:
# ✅ ONE CELL: save model + tokenizer + config + meta to Google Drive

import os, json, torch

# 0) Where to save (clean folder)
EXPORT_DIR = "/content/drive/MyDrive/mini_llm_project/export_100m"
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1) Save model weights (the actual trained parameters)
MODEL_PATH = os.path.join(EXPORT_DIR, "model_weights.pt")
torch.save(built_model_100m.state_dict(), MODEL_PATH)

# 2) Save tokenizer
TOKENIZER_PATH = os.path.join(EXPORT_DIR, "tokenizer.json")
tokenizer.save(TOKENIZER_PATH)

# 3) Save model config (so you can rebuild same architecture later)
CONFIG_PATH = os.path.join(EXPORT_DIR, "model_config.json")
config = {
    "block_size": block_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "dropout": dropout,
    "vocab_size": tokenizer.get_vocab_size()
}
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

# 4) Save meta info (best loss, step, name) - ckpt is optional
META_PATH = os.path.join(EXPORT_DIR, "model_meta.json")
meta = {
    "model_name": "MiniGPT-100M-TinyStories",
    "training_step": int(ckpt["step"]) if "ckpt" in globals() and ckpt is not None and "step" in ckpt else None,
    "best_val_loss": float(ckpt["best_val"]) if "ckpt" in globals() and ckpt is not None and "best_val" in ckpt else None
}
with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print("✅ Saved everything to:", EXPORT_DIR)
print("   -", MODEL_PATH)
print("   -", TOKENIZER_PATH)
print("   -", CONFIG_PATH)
print("   -", META_PATH)

✅ Saved everything to: /content/drive/MyDrive/mini_llm_project/export_100m
   - /content/drive/MyDrive/mini_llm_project/export_100m/model_weights.pt
   - /content/drive/MyDrive/mini_llm_project/export_100m/tokenizer.json
   - /content/drive/MyDrive/mini_llm_project/export_100m/model_config.json
   - /content/drive/MyDrive/mini_llm_project/export_100m/model_meta.json


In [34]:
import os

best_path = "/content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt"

print("Exists:", os.path.exists(best_path))

Exists: True


In [35]:
ckpt = torch.load(best_path, map_location=device)

built_model_100m.load_state_dict(ckpt["model_state"])

built_model_100m.eval()

print("Loaded step:", ckpt["step"])
print("Best val:", ckpt["best_val"])

Loaded step: 48000
Best val: 2.3162708830833436


In [36]:
# import os, torch, math

# PROJECT_PATH = "/content/drive/MyDrive/mini_llm_project"
# BEST_PATH = os.path.join(PROJECT_PATH, "checkpoints_built_100m", "BEST.pt")

# ckpt = torch.load(BEST_PATH, map_location=device)

# built_model_100m.load_state_dict(ckpt["model_state"])
# built_model_100m.to(device)
# built_model_100m.eval()

# print("Loaded BEST from:", BEST_PATH)
# print("BEST step:", ckpt.get("step"))
# print("BEST val_loss (stored):", ckpt.get("best_val"))
# if ckpt.get("best_val") is not None:
#     print("BEST val_ppl (computed):", math.exp(ckpt["best_val"]))

In [37]:
# val_loss, val_ppl = eval_built(built_model_100m, val_loader_100m, eval_iters=200)
# print("Fresh Val Loss:", val_loss)
# print("Fresh Val PPL :", val_ppl)

In [38]:
import torch.nn.functional as F
import torch

@torch.no_grad()
def generate_built(model, start_text, max_new_tokens=140, temperature=0.9, top_k=50):
    model.eval()
    ids = tokenizer.encode(start_text).ids
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        x_cond = x[:, -block_size:]
        logits = model(input_ids=x_cond).logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)
        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist())

In [39]:
prompts = [
    "Once upon a time",
    "In a small village",
    "The little robot"
]

for p in prompts:
    print("\n" + "="*60)
    print("PROMPT:", p)
    print(generate_built(built_model_100m, p))


PROMPT: Once upon a time
Once upon a time in small , family They going an . boy his was years . was excited He going pack luggage his with clothes hats Before . mom dad his and said to boy " don t , ' just off We ." boy his were , he ' getting and in . all his packed bags his so could them . the gave boy bag unpack his . were for trip The was and . had of to careful them . boy out was with the , he to sure all the were in for trip His One boy a smile his . said " you re brave , !" his were happy said you re . family the were happy and had great . all to on way , boy proud have brave . they had a trip Once a

PROMPT: In a small village
In a small village there a lion Leo a , was very . roared roared and in . the in village scared little . the saw and to help other who in village was . lion to a man was a . man a was to a who not of forest and Leo very . wanted be , he to the of animals wanted protect . he to to them asked help they in forest The had idea He Leo he do work he and them t

In [53]:
import torch

def generate_clean(model, tokenizer, prompt,
                   max_new_tokens=140,
                   temperature=0.7,
                   top_p=0.9,
                   top_k=50,
                   repetition_penalty=1.15):
    model.eval()
    device = next(model.parameters()).device

    # tokenizers.Tokenizer -> encode -> ids
    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        out_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repetition_penalty=repetition_penalty,
            pad_token_id=0,     # safe default (we’ll also set EOS below if you have one)
            eos_token_id=None   # keep going until max_new_tokens
        )

    return tokenizer.decode(out_ids[0].tolist())

In [54]:
prompts = ["Once upon a time", "In a small village", "The little robot"]
for p in prompts:
    print("\n" + "="*60)
    print("PROMPT:", p)
    print(generate_clean(built_model_100m, tokenizer, p))


PROMPT: Once upon a time
Once upon a time there a named was little called . day wanted go the and something . went an . was . looked and , it very . saw big in sky It so . was happy Suddenly it to . was from sky The started move It to dark But soon was ! little felt and . was . little scared it not like other . then something happened A came . was bright It like rain The made laugh . little ran home tell parents about adventure They seen Mummy Daddy the . were excited They to him ! said " ' going be !" little smiled they . held mum s tightly As walked , felt wind It them . and laughed they until reached ' side Mummy Daddy s . said Let s home but if ever back you

PROMPT: In a small village
In a small village there an man his was to very . had big and things but was bit . day he to for walk the to market He very . had lot money buy , no what wanted The asked little to some people The said " can a !" man to store bought . he them so ! went home started his . put in of items looked and .

In [55]:
v = tokenizer.get_vocab()
for t in ["</s>", "<eos>", "[EOS]", "<|endoftext|>", "<pad>", "[PAD]"]:
    print(t, v.get(t))

</s> None
<eos> None
[EOS] 3
<|endoftext|> None
<pad> None
[PAD] 0


In [62]:
@torch.no_grad()
def generate_strict(model, tokenizer, prompt, max_new_tokens=120):
    model.eval()
    device = next(model.parameters()).device

    vocab = tokenizer.get_vocab()
    pad_id = vocab.get("[PAD]", 0)
    eos_id = vocab.get("[EOS]", None)   # you showed [EOS] = 3

    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    out = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,

        # IMPORTANT: turn OFF sampling
        do_sample=False,

        # Beam search (more “safe/clean”)
        num_beams=3,
        early_stopping=True,

        repetition_penalty=1.25,
        no_repeat_ngram_size=4,

        eos_token_id=eos_id,
        pad_token_id=pad_id,
    )

    text = tokenizer.decode(out[0].tolist())
    text = " ".join(text.split())  # basic cleanup only
    return text

In [63]:
prompts = ["Once upon a time", "In a small village", "The little robot"]
for p in prompts:
    print("\nPROMPT:", p)
    print(generate_strict(built_model_100m, tokenizer, p))


PROMPT: Once upon a time
Once upon a time there a , was little named . was three old loved explore world One , decided go an . wanted explore world so decided take walk the . walked walked the and a came a . saw big and things It very . saw trees flowers trees flowers trees birds bugs birds and . was happy see things As walked the , noticed strange . saw big , clouds in sky It a . was and , it like was . little was excited explore world Suddenly it to . , heard voice It a voice It , Hello The said " ! ' here I m . you re !" little looked and a scared . voice , What

PROMPT: In a small village
In a small village there a boy Tim Tim Tim Tim a . liked play with toy . day he a . toy a . toy shoot a . was and . played the all . day Tim a named came the . wanted play the too Tim s . said " " " " , I to the ." did want play the . said No it . was . was . did want play Tim s . was . ' mom , was . said " , is nice Tim Tim Tim But need be ." did listen Tim s . played the . shot Tim s . got . wa

In [64]:
for i in range(5):
    print("----")
    print(dataset["train"][i]["text"][:500])

----
Once upon a time, there was a little dog named Max. Max had a hurt leg. He could not run or play with his friends. Max was sad. He looked at the distant park, where his friends were playing.
One day, Max saw a big bird. The bird said, "Max, I can help you. I will include you in my flying fun." Max was happy. He wanted to fly with the bird.
Max and the bird started to fly. They flew high in the sky. Max felt happy and free. But then, the bird let go of Max. Max fell to the ground. The bird said, 
----
Once upon a time, there was a little boy named Tom. Tom loved to run. He wanted to improve and be the best runner. One day, his mom told him about a big race. The winner would get a shiny trophy. Tom was excited and wanted to win.
Tom knew he needed help. He asked his friend, Sam, to be his coach. Sam was reliable and always there for Tom. They practiced every day. Tom got better and better. Sam said, "You will do great in the race!"
On the day of the race, Tom was ready. He remembere

In [41]:
import re
import torch

def clean_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([,.!?;:])", r"\1", s)          # remove space before punctuation
    s = re.sub(r"([,.!?;:])(?=\w)", r"\1 ", s)      # ensure space after punctuation
    s = re.sub(r"(\.\s*){3,}", "... ", s)           # collapse many dots
    s = re.sub(r"\s+'\s*", "'", s)                  # fix apostrophes spacing
    return s.strip()

@torch.no_grad()
def generate_strict(model, tokenizer, prompt,
                    max_new_tokens=120,
                    temperature=0.7,
                    top_k=30,
                    repetition_penalty=1.25,
                    no_repeat_ngram_size=4):
    model.eval()
    device = next(model.parameters()).device

    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    # ✅ Use EOS you actually have: [EOS]
    eos_id = tokenizer.token_to_id("[EOS]")
    if eos_id is None:
        eos_id = None  # if missing, we just won't stop early

    out = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,              # ✅ IMPORTANT: turn OFF sampling (more stable)
        num_beams=3,                  # ✅ beam search helps quality
        early_stopping=True,
        temperature=temperature,      # not used when do_sample=False but ok
        top_k=top_k,                  # not used when do_sample=False but ok
        repetition_penalty=repetition_penalty,
        no_repeat_ngram_size=no_repeat_ngram_size,
        eos_token_id=eos_id,
        pad_token_id=0
    )

    text = tokenizer.decode(out[0].tolist())
    return clean_text(text)

prompts = ["Once upon a time", "In a small village", "The little robot"]
for p in prompts:
    print("\n" + "="*60)
    print("PROMPT:", p)
    print(generate_strict(built_model_100m, tokenizer, p))


PROMPT: Once upon a time
Once upon a time there a, was little named. was three old loved explore world One, decided go an. wanted explore world so decided take walk the. walked walked the and a came a. saw big and things It very. saw trees flowers trees flowers trees birds bugs birds and. was happy see things As walked the, noticed strange. saw big, clouds in sky It a. was and, it like was. little was excited explore world Suddenly it to., heard voice It a voice It, Hello The said "!'here I m. you re!" little looked and a scared. voice, What

PROMPT: In a small village
In a small village there a boy Tim Tim Tim Tim a. liked play with toy. day he a. toy a. toy shoot a. was and. played the all. day Tim a named came the. wanted play the too Tim s. said " " " ", I to the." did want play the. said No it. was. was. did want play Tim s. was.'mom, was. said ", is nice Tim Tim Tim But need be." did listen Tim s. played the. shot Tim s. got. was. ran. was. then Tim hurt leg He.

PROMPT: The lit

# I need to retrain the model with a god tekonizer

In [42]:
# ========= OPTION C: New "clean" tokenizer + save to Drive (ONE CELL) =========
!pip -q install tokenizers datasets

import os, json, random
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.processors import TemplateProcessing

# 0) Paths (clean folder so you won't get confused later)
BASE_DIR = "/content/drive/MyDrive/mini_llm_project"
EXPORT_DIR = os.path.join(BASE_DIR, "artifacts_100m_v2")   # <-- new folder
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1) Load dataset (100M_1)
dataset_100m = load_dataset("eminorhan/tinystories", "100M_1")
train_texts = dataset_100m["train"]["text"]

# 2) Train ByteLevel BPE tokenizer (GPT-style)
# vocab_size: 16k-32k is good. Try 16k first (faster).
vocab_size = 16000

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)

# Train on a subset for speed (still works well). Increase if you want.
# Using 200k examples is usually enough to learn punctuation/spacing well.
sample_size = min(200_000, len(train_texts))
sample_texts = random.sample(train_texts, sample_size)

tokenizer.train_from_iterator(sample_texts, trainer=trainer)

# Make tokenizer always add BOS/EOS
bos_id = tokenizer.token_to_id("[BOS]")
eos_id = tokenizer.token_to_id("[EOS]")
tokenizer.post_processor = TemplateProcessing(
    single=f"[BOS] $A [EOS]",
    pair=f"[BOS] $A [EOS] $B:1 [EOS]:1",
    special_tokens=[("[BOS]", bos_id), ("[EOS]", eos_id)]
)

pad_id = tokenizer.token_to_id("[PAD]")

# 3) Save tokenizer files to Drive
tokenizer_json_path = os.path.join(EXPORT_DIR, "tokenizer.json")
tokenizer.save(tokenizer_json_path)

tokenizer_meta = {
    "vocab_size": vocab_size,
    "pad_token": "[PAD]",
    "unk_token": "[UNK]",
    "bos_token": "[BOS]",
    "eos_token": "[EOS]",
    "pad_id": pad_id,
    "bos_id": bos_id,
    "eos_id": eos_id,
    "dataset": "eminorhan/tinystories 100M_1"
}
with open(os.path.join(EXPORT_DIR, "tokenizer_meta.json"), "w") as f:
    json.dump(tokenizer_meta, f, indent=2)

print("✅ Saved tokenizer to:", tokenizer_json_path)
print("✅ Saved meta to:", os.path.join(EXPORT_DIR, "tokenizer_meta.json"))

# 4) Quick sanity test: encode/decode should keep punctuation clean
tests = [
    "Once upon a time, there was a robot.",
    "In a small village, Tim's toy broke! He cried.",
    "Hello... why is spacing weird? It shouldn't be."
]
print("\n--- SANITY TEST (encode -> decode) ---")
for t in tests:
    ids = tokenizer.encode(t).ids
    back = tokenizer.decode(ids)
    print("\nIN :", t)
    print("OUT:", back)

✅ Saved tokenizer to: /content/drive/MyDrive/mini_llm_project/artifacts_100m_v2/tokenizer.json
✅ Saved meta to: /content/drive/MyDrive/mini_llm_project/artifacts_100m_v2/tokenizer_meta.json

--- SANITY TEST (encode -> decode) ---

IN : Once upon a time, there was a robot.
OUT:  Once upon a time, there was a robot.

IN : In a small village, Tim's toy broke! He cried.
OUT:  In a small village, Tim's toy broke! He cried.

IN : Hello... why is spacing weird? It shouldn't be.
OUT:  Hello... why is spacing weird? It shouldn't be.


In [43]:
# ========= STEP 2: Retokenize + Chunk to fixed block_size + DataLoaders (ONE CELL) =========
import os, math, random
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer

# 0) Load tokenizer from Drive
EXPORT_DIR = "/content/drive/MyDrive/mini_llm_project/artifacts_100m_v2"
tokenizer = Tokenizer.from_file(os.path.join(EXPORT_DIR, "tokenizer.json"))

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("PAD_ID:", PAD_ID, "BOS_ID:", BOS_ID, "EOS_ID:", EOS_ID)

# 1) Load dataset (100M_1)
dataset_100m = load_dataset("eminorhan/tinystories", "100M_1")

# 2) Tokenize each story -> ids
def tok_batch(batch):
    ids_list = []
    for t in batch["text"]:
        ids = tokenizer.encode(t).ids
        ids_list.append(ids)
    return {"ids": ids_list}

# ⚡ speed tip: you can start with a small subset first to test (then remove select later)
# dataset_100m = dataset_100m.select_columns(["text"])
# dataset_100m["train"] = dataset_100m["train"].select(range(20000))
# dataset_100m["validation"] = dataset_100m["validation"].select(range(2000))

tokd = dataset_100m.map(tok_batch, batched=True, remove_columns=["text"])

# 3) Chunk into fixed blocks for causal LM
block_size = 256  # good default

def group_texts(examples):
    # flatten list of lists
    concatenated = []
    for x in examples["ids"]:
        concatenated.extend(x)

    # keep only full blocks (block_size + 1 for next-token labels)
    total_length = (len(concatenated) // (block_size + 1)) * (block_size + 1)
    concatenated = concatenated[:total_length]

    input_ids = []
    labels = []

    for i in range(0, total_length, block_size + 1):
        chunk = concatenated[i : i + block_size + 1]
        x = chunk[:-1]
        y = chunk[1:]
        input_ids.append(x)
        labels.append(y)

    return {"input_ids": input_ids, "labels": labels}

lm_100m = tokd.map(group_texts, batched=True, remove_columns=["ids"])

print(lm_100m)

# 4) Set format for PyTorch
lm_100m.set_format(type="torch", columns=["input_ids", "labels"])

# 5) DataLoaders (NOW every sample same length => no more "equal size" error)
batch_size = 8  # if OOM, switch to 4

train_loader_100m = DataLoader(lm_100m["train"], batch_size=batch_size, shuffle=True)
val_loader_100m   = DataLoader(lm_100m["validation"], batch_size=batch_size, shuffle=False)

# 6) Quick check: shape should be [B, block_size]
xb = next(iter(train_loader_100m))["input_ids"]
yb = next(iter(train_loader_100m))["labels"]
print("✅ Batch input shape:", xb.shape)
print("✅ Batch label shape:", yb.shape)

PAD_ID: 0 BOS_ID: 2 EOS_ID: 3


Map:   0%|          | 0/622827 [00:00<?, ? examples/s]

Map:   0%|          | 0/27635 [00:00<?, ? examples/s]

Map:   0%|          | 0/622827 [00:00<?, ? examples/s]

Map:   0%|          | 0/27635 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 477925
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 21068
    })
})
✅ Batch input shape: torch.Size([8, 256])
✅ Batch label shape: torch.Size([8, 256])


In [45]:
# =========================
# FULL TRAIN + EVAL + SAVE
# =========================
import os, time, math
import torch
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

# -------------------------
# 0) REQUIRED THINGS YOU ALREADY HAVE
# -------------------------
# model = built_model_100m
# train_loader_100m, val_loader_100m
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = built_model_100m.to(device)

# -------------------------
# 1) PATHS
# -------------------------
PROJECT_PATH = "/content/drive/MyDrive/mini_llm_project"
CKPT_DIR = os.path.join(PROJECT_PATH, "checkpoints_built_100m")
os.makedirs(CKPT_DIR, exist_ok=True)

BEST_PATH = os.path.join(CKPT_DIR, "BEST.pt")
LAST_PATH = os.path.join(CKPT_DIR, "LAST.pt")

# -------------------------
# 2) SETTINGS
# -------------------------
target_steps = 50000
print_every = 200
eval_every  = 2000
eval_iters  = 200

lr = 3e-4
weight_decay = 0.1
grad_accum_steps = 4
max_grad_norm = 1.0
use_amp = True

# -------------------------
# 3) OPTIMIZER + AMP
# -------------------------
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type=="cuda"))

# -------------------------
# 4) OPTIONAL: RESUME FROM BEST (if exists)
# -------------------------
start_step = 0
best_val = float("inf")

if os.path.exists(BEST_PATH):
    ckpt = torch.load(BEST_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    if "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    start_step = int(ckpt.get("step", 0))
    best_val = float(ckpt.get("best_val", best_val))
    print(f"✅ Loaded BEST checkpoint from {BEST_PATH}")
    print("   step:", start_step, " best_val:", best_val)
else:
    ckpt = {}  # so code doesn't crash later
    print("ℹ️ No BEST checkpoint found. Training from scratch.")

# -------------------------
# 5) SCHEDULER (cosine with warmup)
# NOTE: Scheduler steps only when optimizer.step happens
# -------------------------
total_updates = target_steps // grad_accum_steps
warmup_updates = max(1, int(0.05 * total_updates))  # 5% warmup

done_updates = start_step // grad_accum_steps
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_updates,
    num_training_steps=total_updates
)

# fast-forward scheduler if resuming
for _ in range(done_updates):
    scheduler.step()

print("✅ Scheduler ready:",
      "total_updates=", total_updates,
      "warmup_updates=", warmup_updates,
      "done_updates=", done_updates)

# -------------------------
# 6) EVALUATION FUNCTION
# -------------------------
@torch.no_grad()
def eval_model(model, val_loader, iters=200):
    model.eval()
    losses = []
    val_iter = iter(val_loader)

    for _ in range(iters):
        try:
            batch = next(val_iter)
        except StopIteration:
            val_iter = iter(val_loader)
            batch = next(val_iter)

        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)

        out = model(input_ids=x, labels=y)   # ✅ correct
        loss = out.loss
        losses.append(loss.item())

    model.train()
    avg_loss = sum(losses) / len(losses)
    ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    return avg_loss, ppl

# -------------------------
# 7) TRAIN LOOP
# -------------------------
model.train()
train_iter = iter(train_loader_100m)

t0 = time.time()
running_loss = 0.0

for step in range(start_step, target_steps):
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader_100m)
        batch = next(train_iter)

    x = batch["input_ids"].to(device)
    y = batch["labels"].to(device)

    with torch.cuda.amp.autocast(enabled=(use_amp and device.type=="cuda")):
        out = model(input_ids=x, labels=y)          # ✅ FIXED: keyword args
        loss = out.loss / grad_accum_steps

    scaler.scale(loss).backward()
    running_loss += loss.item()

    # optimizer step every grad_accum_steps
    if (step + 1) % grad_accum_steps == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        # ✅ scheduler AFTER optimizer.step
        scheduler.step()

    # print
    if (step + 1) % print_every == 0:
        elapsed = time.time() - t0
        avg_train_loss = running_loss / print_every
        running_loss = 0.0

        lr_now = optimizer.param_groups[0]["lr"]
        steps_per_sec = print_every / max(elapsed, 1e-9)
        print(f"[100m] step {step+1}/{target_steps} | train_loss {avg_train_loss:.4f} | lr {lr_now:.2e} | {steps_per_sec:.2f} steps/s")
        t0 = time.time()

    # eval + save
    if (step + 1) % eval_every == 0:
        val_loss, val_ppl = eval_model(model, val_loader_100m, iters=eval_iters)
        print(f"   🔎 VAL | step {step+1} | val_loss {val_loss:.4f} | ppl {val_ppl:.2f}")

        # always save LAST
        torch.save({
            "step": step + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val": best_val,
        }, LAST_PATH)
        print("   💾 Saved LAST:", LAST_PATH)

        # save BEST
        if val_loss < best_val:
            best_val = val_loss
            torch.save({
                "step": step + 1,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val": best_val,
            }, BEST_PATH)
            print("   ✅ New BEST saved:", BEST_PATH)

print("✅ Done training.")
print("BEST val loss:", best_val)
print("BEST checkpoint:", BEST_PATH)
print("LAST checkpoint:", LAST_PATH)

/tmp/ipykernel_3142/3210547977.py:46: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type=="cuda"))


✅ Loaded BEST checkpoint from /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt
   step: 48000  best_val: 2.3162708830833436
✅ Scheduler ready: total_updates= 12500 warmup_updates= 625 done_updates= 12000


/tmp/ipykernel_3142/3210547977.py:83: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
/tmp/ipykernel_3142/3210547977.py:137: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type=="cuda")):


[100m] step 48200/50000 | train_loss 2.1144 | lr 1.06e-06 | 48.68 steps/s
[100m] step 48400/50000 | train_loss 1.9555 | lr 8.39e-07 | 50.16 steps/s
[100m] step 48600/50000 | train_loss 1.8732 | lr 6.43e-07 | 50.90 steps/s
[100m] step 48800/50000 | train_loss 1.8265 | lr 4.72e-07 | 50.99 steps/s
[100m] step 49000/50000 | train_loss 1.7965 | lr 3.28e-07 | 51.00 steps/s
[100m] step 49200/50000 | train_loss 1.7788 | lr 2.10e-07 | 51.04 steps/s
[100m] step 49400/50000 | train_loss 1.7676 | lr 1.18e-07 | 51.06 steps/s
[100m] step 49600/50000 | train_loss 1.7637 | lr 5.25e-08 | 51.05 steps/s
[100m] step 49800/50000 | train_loss 1.7614 | lr 1.31e-08 | 51.06 steps/s
[100m] step 50000/50000 | train_loss 1.7621 | lr 0.00e+00 | 51.05 steps/s
   🔎 VAL | step 50000 | val_loss 7.0148 | ppl 1113.04
   💾 Saved LAST: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/LAST.pt
✅ Done training.
BEST val loss: 2.3162708830833436
BEST checkpoint: /content/drive/MyDrive/mini_llm_project/checkpoint

In [50]:
import torch
from tokenizers import Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- paths ----
BEST_PATH = "/content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt"
TOKENIZER_PATH = "/content/drive/MyDrive/mini_llm_project/artifacts_100m_v2/tokenizer.json"

# ---- load tokenizer ----
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

# ---- load model ----
ckpt = torch.load(BEST_PATH, map_location=device)

model = built_model_100m.to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()

print("Model loaded. Best val loss:", ckpt["best_val"])


# ---- generation function ----
@torch.no_grad()






# ---- generation function ----
@torch.no_grad()
def generate(prompt, max_new_tokens=120, temperature=0.9, top_k=50):

    ids = tokenizer.encode(prompt).ids
    ids = [BOS_ID] + ids

    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):

        logits = model(x)

        next_logits = logits[:, -1, :] / temperature

        v, ix = torch.topk(next_logits, k=top_k)
        probs = torch.softmax(v, dim=-1)

        next_id = ix[:, torch.multinomial(probs, 1)]

        if next_id.item() == EOS_ID:
            break

        x = torch.cat([x, next_id], dim=1)

    text = tokenizer.decode(x[0].tolist())
    return text




Model loaded. Best val loss: 2.3162708830833436


In [54]:
s = "Once upon a time, a brave cat walked into a small village."

enc = tokenizer.encode(s)

print("ids:", enc.ids[:20])
print("decoded:", tokenizer.decode(enc.ids))

ids: [2, 282, 292, 111, 250, 12, 111, 908, 312, 640, 567, 111, 412, 1850, 14, 3]
decoded:  Once upon a time, a brave cat walked into a small village.


In [58]:
import os, torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Loading BEST checkpoint from:", BEST_PATH)
assert os.path.exists(BEST_PATH), f"BEST_PATH not found: {BEST_PATH}"

ckpt = torch.load(BEST_PATH, map_location=device)

# Some checkpoints store weights directly, others store dict with 'model_state'
if isinstance(ckpt, dict) and "model_state" in ckpt:
    model.load_state_dict(ckpt["model_state"], strict=True)
elif isinstance(ckpt, dict) and "model" in ckpt:
    model.load_state_dict(ckpt["model"], strict=True)
else:
    # If you saved state_dict directly
    model.load_state_dict(ckpt, strict=True)

model.eval()
print("✅ BEST checkpoint loaded and model set to eval()")

Loading BEST checkpoint from: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt
✅ BEST checkpoint loaded and model set to eval()


In [59]:
import torch
import torch.nn.functional as F

def _top_p_filtering(logits, top_p=0.9):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    probs = F.softmax(sorted_logits, dim=-1)
    cumprobs = torch.cumsum(probs, dim=-1)

    # remove tokens with cumulative probability above threshold
    mask = cumprobs > top_p
    mask[..., 0] = False  # keep at least 1 token
    sorted_logits[mask] = float("-inf")

    # unsort back
    unsorted = torch.full_like(logits, float("-inf"))
    unsorted.scatter_(1, sorted_indices, sorted_logits)
    return unsorted

@torch.no_grad()
def generate_text(
    prompt,
    model,
    tokenizer,
    max_new_tokens=120,
    temperature=0.7,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.15,
    eos_token_id=3,
):
    model.eval()
    device = next(model.parameters()).device

    enc = tokenizer.encode(prompt)
    input_ids = torch.tensor([enc.ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        outputs = model(input_ids)

        if hasattr(outputs, "logits"):
            logits = outputs.logits
        else:
            logits = outputs[0]

        logits = logits[:, -1, :]  # [1, vocab]

        # repetition penalty (discourage repeating recent tokens)
        if repetition_penalty is not None and repetition_penalty > 1.0:
            recent = input_ids[0, -128:]  # last 128 tokens
            logits[0, recent] = logits[0, recent] / repetition_penalty

        # temperature
        if temperature and temperature > 0:
            logits = logits / temperature

        # top-k
        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            filtered = torch.full_like(logits, float("-inf"))
            filtered.scatter_(1, ix, v)
            logits = filtered

        # top-p
        if top_p and 0 < top_p < 1:
            logits = _top_p_filtering(logits, top_p=top_p)

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_id], dim=1)

        if next_id.item() == eos_token_id:
            break

    return tokenizer.decode(input_ids[0].tolist())


# QUICK TEST
prompts = [
    "Once upon a time",
    "In a small village",
    "The little robot",
    "One day Tim found",
    "A brave cat",
]

print("\n================ GENERATION TEST ================\n")
for p in prompts:
    print("PROMPT:", p)
    print(generate_text(p, model, tokenizer))
    print("\n" + "-" * 60 + "\n")


================ GENERATION TEST ================

PROMPT: Once upon a time
 Once upon a timeay th. n
� purpleis� played� M Lucy The kind. n played� kind� theyver,
 me�- t H clo! she youriend_ hel weess_. mom��_ haionle yell�on" it�.Gk� itG itun helst.ownownist_ tMom, Sheingowack. She"heron hel"herss care_ get never She_�owri� outilly�&aut amaz lst Timirrel At l too. hel But what ch hadNor� watching B�

------------------------------------------------------------

PROMPT: In a small village
 In a small village�,_on",&� wheraxdy�ri star�You doing livedfortableirrelillyle��. tump ro v�The., She�ared plantsing, c, herird�ver n wh Shehedque. t�hing_ fox wor his shole ha eucyay her str! uri cook_ cook police be, nice�!ck t r Shest. him� se down policeug�� tow prinankel,Nor had. u v. she youriend_ helld th. weimexpected She� me.

------------------------------------------------------------

PROMPT: The little robot
 The little robot�ing._ Timing t�hed She&thingque.,axon" it�. wa itentherure

#  Today WHere I started

In [13]:
from tokenizers import Tokenizer

TOKENIZER_PATH = "/content/drive/MyDrive/mini_llm_project/export_100m/tokenizer.json"
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

EOS_ID = tokenizer.token_to_id("[EOS]")
print("✅ Tokenizer loaded:", TOKENIZER_PATH)
print("vocab_size =", tokenizer.get_vocab_size())
print("EOS_ID =", EOS_ID)

✅ Tokenizer loaded: /content/drive/MyDrive/mini_llm_project/export_100m/tokenizer.json
vocab_size = 16000
EOS_ID = 3


In [15]:
from transformers import GPT2Config, GPT2LMHeadModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = tokenizer.get_vocab_size()
block_size = 256
n_embd = 768
n_head = 12
n_layer = 12
dropout = 0.1

cfg = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=n_embd,
    n_layer=n_layer,
    n_head=n_head,
    resid_pdrop=dropout,
    embd_pdrop=dropout,
    attn_pdrop=dropout,
)

model = GPT2LMHeadModel(cfg).to(device)
model.eval()

print("✅ Model created. model vocab =", model.config.vocab_size)

✅ Model created. model vocab = 16000


In [16]:
import os, torch

BEST_PATH = "/content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt"
assert os.path.exists(BEST_PATH), f"BEST checkpoint not found: {BEST_PATH}"

ckpt = torch.load(BEST_PATH, map_location=device)

if isinstance(ckpt, dict) and "model_state" in ckpt:
    model.load_state_dict(ckpt["model_state"], strict=True)
elif isinstance(ckpt, dict) and "model" in ckpt:
    model.load_state_dict(ckpt["model"], strict=True)
else:
    model.load_state_dict(ckpt, strict=True)

model.eval()
print("✅ BEST checkpoint loaded:", BEST_PATH)

✅ BEST checkpoint loaded: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt


In [17]:
s = "Once upon a time, a brave cat walked into a small village."
enc = tokenizer.encode(s)
print("decoded:", tokenizer.decode(enc.ids))  # should look normal

decoded: Once upon a time , a brave cat walked into a small village .


In [19]:
import torch.nn.functional as F
import torch

@torch.no_grad()
def generate_text(prompt, model, tokenizer, max_new_tokens=120, temperature=0.6, top_k=20, eos_token_id=None):
    model.eval()
    device = next(model.parameters()).device

    enc = tokenizer.encode(prompt)
    input_ids = torch.tensor([enc.ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        outputs = model(input_ids)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]
        logits = logits[:, -1, :]

        if temperature and temperature > 0:
            logits = logits / temperature

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            filtered = torch.full_like(logits, float("-inf"))
            filtered.scatter_(1, ix, v)
            logits = filtered

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_id], dim=1)

        if eos_token_id is not None and next_id.item() == eos_token_id:
            break

    return tokenizer.decode(input_ids[0].tolist())

In [20]:
print("EOS_ID =", EOS_ID)
print(generate_text("Once upon a time", model, tokenizer, max_new_tokens=120, temperature=0.6, top_k=20, eos_token_id=EOS_ID))

EOS_ID = 3
Once upon a time there a girl Jenny She three old loved explore One she a and parents her were the of . was excited she going the to the . Jenny a three old was to on journey She to beach When arrived she the was excited She her . saw big , shells the and shells She never before She to them She to . asked parents she to them they , they she , they yes When arrived the was big . were . was a scared and was for first . was bit but was . felt and . was to a , beach Jenny . saw waves the and waves She so . looked the and it so !


In [21]:
print(generate_text(
    "Once upon a time",
    model,
    tokenizer,
    max_new_tokens=80,
    temperature=0.4,
    top_k=10,
    eos_token_id=EOS_ID
))

Once upon a time there a girl Daisy to in small . was very and liked play her . day Daisy playing her and fun to . , was her friend s . friend a , , , , , , , , , , , , , , , , , , . liked play Daisy s too Daisy They together day Daisy s . day Daisy a for . wanted play game She to . said " ' ', ' !' said


In [22]:
prompts = [
    "Once upon a time",
    "There was a little girl named Jenny",
    "Tim and Ben liked to play"
]

for p in prompts:
    print("\nPROMPT:", p)
    print(generate_text(
        p,
        model,
        tokenizer,
        max_new_tokens=80,
        temperature=0.4,
        top_k=10,
        eos_token_id=EOS_ID
    ))
    print("-" * 60)


PROMPT: Once upon a time
Once upon a time there a boy Jack He three old was in room with mom He looking something . was a . was very and was excited he to out window He see was . saw big and things the was . saw big and things He to . saw , and things He saw He a . was . wanted touch , he to what was . mom , , , was . said " , , , , , , ,
------------------------------------------------------------

PROMPT: There was a little girl named Jenny
There was a little girl named Jenny She very . was years and loved explore One , wanted explore world her . went a and day found big in woods She so that decided explore . she a and adventure the was . she an and creature a . was and and , she very . she to a of , she a , animal It a . was and , it a , animal It very . was and , it a . was and ,
------------------------------------------------------------

PROMPT: Tim and Ben liked to play
Tim and Ben liked to play the . liked pretend were and , they to they in park One , saw big in park It a . w

## What I should do

In [1]:
import torch
print (torch.__version__)
print (torch.cuda.is_available())
print("GPU Namem: ", torch.cuda.get_device_name(0))

2.10.0+cu128
True
GPU Namem:  NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
PROJECT_PATH = "/content/drive/MyDrive/mini_llm_project"
CKPT_DIR = f"{PROJECT_PATH}/checkpoints_built_100m"
BEST_PATH = f"{CKPT_DIR}/BEST.pt"
LAST_PATH = f"{CKPT_DIR}/LAST.pt"

In [3]:
from tokenizers import Tokenizer

TOKENIZER_PATH = "/content/drive/MyDrive/mini_llm_project/export_100m/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

EOS_ID = tokenizer.token_to_id("[EOS]")

print("Tokenizer loaded")
print("vocab size:", tokenizer.get_vocab_size())
print("EOS:", EOS_ID)

Tokenizer loaded
vocab size: 16000
EOS: 3


In [4]:
from datasets import load_dataset

dataset_100m = load_dataset("eminorhan/tinystories", "100M_1")
print(dataset_100m)
print(dataset_100m["train"][0]["text"][:200])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

100M_1/train-00000-of-00002.parquet:   0%|          | 0.00/130M [00:00<?, ?B/s]

100M_1/train-00001-of-00002.parquet:   0%|          | 0.00/130M [00:00<?, ?B/s]

100M_1/validation-00000-of-00001.parquet:   0%|          | 0.00/11.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/622827 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/27635 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 622827
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 27635
    })
})
Once upon a time, there was a little boy named Tim. Tim had a toy bear named Ben. Tim and Ben liked to play all day. They would run, jump, and take big steps together. They were best friends.
One day,


In [5]:
def tokenize_with_eos(examples):
    ids_list = []
    for text in examples["text"]:
        ids = tokenizer.encode(text).ids
        ids.append(EOS_ID)
        ids_list.append(ids)
    return {"ids": ids_list}

tok_100m = dataset_100m.map(
    tokenize_with_eos,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/622827 [00:00<?, ? examples/s]

Map:   0%|          | 0/27635 [00:00<?, ? examples/s]

In [6]:
block_size = 256

def make_blocks(examples):
    input_ids = []
    labels = []

    for ids in examples["ids"]:
        # go through one story at a time
        for i in range(0, len(ids) - 1, block_size):
            chunk = ids[i : i + block_size + 1]

            # need at least 2 tokens to make x and y
            if len(chunk) < 2:
                continue

            x = chunk[:-1]   # input
            y = chunk[1:]    # next-token labels

            input_ids.append(x)
            labels.append(y)

    return {
        "input_ids": input_ids,
        "labels": labels
    }

lm_100m = tok_100m.map(
    make_blocks,
    batched=True,
    remove_columns=["ids"]
)

print("✅ lm_100m created")
print(lm_100m)
print(lm_100m["train"][0].keys())
print("Example input length:", len(lm_100m["train"][0]["input_ids"]))
print("Example label length:", len(lm_100m["train"][0]["labels"]))

Map:   0%|          | 0/622827 [00:00<?, ? examples/s]

Map:   0%|          | 0/27635 [00:00<?, ? examples/s]

✅ lm_100m created
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 698035
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 30847
    })
})
dict_keys(['input_ids', 'labels'])
Example input length: 181
Example label length: 181


In [7]:
block_size = 256

def make_blocks(examples):
    input_ids = []
    labels = []

    for ids in examples["ids"]:
        # only keep chunks that are big enough for full x/y length 256
        for i in range(0, len(ids) - (block_size + 1) + 1, block_size):
            chunk = ids[i : i + block_size + 1]

            if len(chunk) == block_size + 1:
                x = chunk[:-1]   # length 256
                y = chunk[1:]    # length 256

                input_ids.append(x)
                labels.append(y)

    return {
        "input_ids": input_ids,
        "labels": labels
    }

lm_100m = tok_100m.map(
    make_blocks,
    batched=True,
    remove_columns=["ids"]
)

print("✅ lm_100m created")
print(lm_100m)
print(lm_100m["train"][0].keys())
print("Example input length:", len(lm_100m["train"][0]["input_ids"]))
print("Example label length:", len(lm_100m["train"][0]["labels"]))

Map:   0%|          | 0/622827 [00:00<?, ? examples/s]

Map:   0%|          | 0/27635 [00:00<?, ? examples/s]

✅ lm_100m created
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 76031
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 3246
    })
})
dict_keys(['input_ids', 'labels'])
Example input length: 256
Example label length: 256


In [10]:
from torch.utils.data import DataLoader
import torch

batch_size = 8

def collate_fn(batch):
    input_ids = torch.tensor([item["input_ids"] for item in batch], dtype=torch.long)
    labels = torch.tensor([item["labels"] for item in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "labels": labels
    }

train_loader = DataLoader(
    lm_100m["train"],
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    lm_100m["validation"],
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

print("✅ DataLoaders created")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

batch = next(iter(train_loader))
print("Batch keys:", batch.keys())
print("input_ids shape:", batch["input_ids"].shape)
print("labels shape:", batch["labels"].shape)

✅ DataLoaders created
Train batches: 9504
Validation batches: 406
Batch keys: dict_keys(['input_ids', 'labels'])
input_ids shape: torch.Size([8, 256])
labels shape: torch.Size([8, 256])


In [12]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [14]:
from transformers import GPT2Config, GPT2LMHeadModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = tokenizer.get_vocab_size()
block_size = 256
n_embd = 768
n_head = 12
n_layer = 12
dropout = 0.1

cfg = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=n_embd,
    n_layer=n_layer,
    n_head=n_head,
    resid_pdrop=dropout,
    embd_pdrop=dropout,
    attn_pdrop=dropout,
)

model = GPT2LMHeadModel(cfg).to(device)
model.eval()

print("✅ Model created")
print("Using device:", device)
print("Model vocab:", model.config.vocab_size)

✅ Model created
Using device: cuda
Model vocab: 16000


In [15]:
import os, torch

start_step = 0
best_val = float("inf")

if os.path.exists(LAST_PATH):
    print("🔁 Resuming from LAST:", LAST_PATH)
    ckpt = torch.load(LAST_PATH, map_location=device)

    # load model
    if "model_state" in ckpt:
        model.load_state_dict(ckpt["model_state"], strict=True)
    elif "model" in ckpt:
        model.load_state_dict(ckpt["model"], strict=True)
    else:
        model.load_state_dict(ckpt, strict=True)

    # optional: resume step + best_val if saved
    start_step = int(ckpt.get("step", 0))
    best_val   = float(ckpt.get("best_val", float("inf")))

    print("✅ Resumed. start_step =", start_step, " best_val =", best_val)
else:
    print("🆕 No LAST checkpoint found. Training from scratch.")

🔁 Resuming from LAST: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/LAST.pt
✅ Resumed. start_step = 50000  best_val = 2.3162708830833436


In [16]:
import torch.nn.functional as F

lr = 3e-4
weight_decay = 0.1
max_grad_norm = 1.0
use_amp = True

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))

/tmp/ipykernel_7151/3738182273.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))


In [17]:
import torch

@torch.no_grad()
def eval_loss(model, loader, eval_batches=200):
    model.eval()
    losses = []
    for i, batch in enumerate(loader):
        if i >= eval_batches:
            break
        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)
        out = model(input_ids=x, labels=y)
        losses.append(out.loss.item())
    model.train()
    return sum(losses) / len(losses)

In [18]:
target_steps = start_step + 30000
print_every = 200
eval_every = 2000
save_every = 2000   # save LAST often so you don't lose progress
grad_accum_steps = 4

In [19]:
import math, time
from contextlib import nullcontext

model.train()
t0 = time.time()

step = start_step

train_iter = iter(train_loader)

while step < target_steps:
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    for _ in range(grad_accum_steps):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)

        autocast_ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if (use_amp and device.type=="cuda") else nullcontext()
        with autocast_ctx:
            out = model(input_ids=x, labels=y)
            loss = out.loss / grad_accum_steps

        scaler.scale(loss).backward()
        total_loss += loss.item()

    # clip grads
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

    scaler.step(optimizer)
    scaler.update()

    step += 1

    if step % print_every == 0:
        dt = time.time() - t0
        print(f"step {step} | train_loss {total_loss:.4f} | time {dt:.1f}s")
        t0 = time.time()

    # eval + save best
    if step % eval_every == 0:
        val = eval_loss(model, val_loader, eval_batches=200)
        print(f"✅ step {step} | val_loss {val:.4f}")

        # save BEST
        if val < best_val:
            best_val = val
            torch.save({
                "model_state": model.state_dict(),
                "step": step,
                "best_val": best_val,
            }, BEST_PATH)
            print("🏆 Saved BEST:", BEST_PATH)

    # always save LAST sometimes
    if step % save_every == 0:
        torch.save({
            "model_state": model.state_dict(),
            "step": step,
            "best_val": best_val,
        }, LAST_PATH)
        print("💾 Saved LAST:", LAST_PATH)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


step 50200 | train_loss 2.3941 | time 16.8s
step 50400 | train_loss 2.4062 | time 15.6s
step 50600 | train_loss 2.4149 | time 15.6s
step 50800 | train_loss 2.3432 | time 15.6s
step 51000 | train_loss 2.4468 | time 15.6s
step 51200 | train_loss 2.2736 | time 15.6s
step 51400 | train_loss 2.2545 | time 15.6s
step 51600 | train_loss 2.3286 | time 15.6s
step 51800 | train_loss 2.2854 | time 15.6s
step 52000 | train_loss 2.3218 | time 15.6s
✅ step 52000 | val_loss 2.2498
🏆 Saved BEST: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt
💾 Saved LAST: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/LAST.pt
step 52200 | train_loss 2.0704 | time 37.5s
step 52400 | train_loss 2.1787 | time 15.6s
step 52600 | train_loss 2.1914 | time 15.6s
step 52800 | train_loss 2.1999 | time 15.6s
step 53000 | train_loss 2.1808 | time 15.6s
step 53200 | train_loss 2.2074 | time 15.6s
step 53400 | train_loss 2.1449 | time 15.6s
step 53600 | train_loss 2.0955 | time 15.6s
step 53

In [20]:
import torch

ckpt = torch.load(BEST_PATH, map_location=device)

if "model_state" in ckpt:
    model.load_state_dict(ckpt["model_state"], strict=True)
elif "model" in ckpt:
    model.load_state_dict(ckpt["model"], strict=True)
else:
    model.load_state_dict(ckpt, strict=True)

model.eval()

print("✅ BEST model loaded")
print("BEST step:", ckpt.get("step", "unknown"))
print("BEST val_loss:", ckpt.get("best_val", "unknown"))

✅ BEST model loaded
BEST step: 64000
BEST val_loss: 2.0251689106225967


In [22]:
import os
import time
import math
import torch
import torch.nn.functional as F
from contextlib import nullcontext

# =========================
# 1) RESUME FROM LAST
# =========================
start_step = 0
best_val = float("inf")

if os.path.exists(LAST_PATH):
    print("🔁 Resuming from LAST:", LAST_PATH)
    ckpt = torch.load(LAST_PATH, map_location=device)

    if "model_state" in ckpt:
        model.load_state_dict(ckpt["model_state"], strict=True)
    elif "model" in ckpt:
        model.load_state_dict(ckpt["model"], strict=True)
    else:
        model.load_state_dict(ckpt, strict=True)

    start_step = int(ckpt.get("step", 0))
    best_val = float(ckpt.get("best_val", float("inf")))

    print("✅ Resumed. start_step =", start_step, " best_val =", best_val)
else:
    print("🆕 No LAST checkpoint found. Training from scratch.")

# =========================
# 2) OPTIMIZER + AMP
# =========================
lr = 3e-4
weight_decay = 0.1
max_grad_norm = 1.0
use_amp = True

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))

# =========================
# 3) EVAL FUNCTION
# =========================
@torch.no_grad()
def eval_loss(model, loader, eval_batches=200):
    model.eval()
    losses = []

    for i, batch in enumerate(loader):
        if i >= eval_batches:
            break

        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)

        out = model(input_ids=x, labels=y)
        losses.append(out.loss.item())

    model.train()
    return sum(losses) / len(losses)

# =========================
# 4) TRAINING SETTINGS
# =========================
target_steps = 150000
print_every = 200
eval_every = 2000
save_every = 2000
grad_accum_steps = 4

print("🚀 Training from step", start_step, "to", target_steps)

# =========================
# 5) TRAIN LOOP
# =========================
model.train()
t0 = time.time()
step = start_step

train_iter = iter(train_loader)

while step < target_steps:
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0

    for _ in range(grad_accum_steps):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)

        autocast_ctx = (
            torch.autocast(device_type="cuda", dtype=torch.float16)
            if (use_amp and device.type == "cuda")
            else nullcontext()
        )

        with autocast_ctx:
            out = model(input_ids=x, labels=y)
            loss = out.loss / grad_accum_steps

        scaler.scale(loss).backward()
        total_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

    scaler.step(optimizer)
    scaler.update()

    step += 1

    if step % print_every == 0:
        dt = time.time() - t0
        print(f"step {step} | train_loss {total_loss:.4f} | time {dt:.1f}s")
        t0 = time.time()

    if step % eval_every == 0:
        val = eval_loss(model, val_loader, eval_batches=200)
        print(f"✅ step {step} | val_loss {val:.4f}")

        if val < best_val:
            best_val = val
            torch.save({
                "model_state": model.state_dict(),
                "step": step,
                "best_val": best_val,
            }, BEST_PATH)
            print("🏆 Saved BEST:", BEST_PATH)

    if step % save_every == 0:
        torch.save({
            "model_state": model.state_dict(),
            "step": step,
            "best_val": best_val,
        }, LAST_PATH)
        print("💾 Saved LAST:", LAST_PATH)

print("🎉 Training finished at step", step)
print("Best val loss:", best_val)

🔁 Resuming from LAST: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/LAST.pt
✅ Resumed. start_step = 80000  best_val = 2.0251689106225967
🚀 Training from step 80000 to 150000


/tmp/ipykernel_7151/1687241859.py:41: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))


step 80200 | train_loss 1.3017 | time 15.8s
step 80400 | train_loss 1.3110 | time 15.6s
step 80600 | train_loss 1.4854 | time 15.6s
step 80800 | train_loss 1.4194 | time 15.6s
step 81000 | train_loss 1.4665 | time 15.6s
step 81200 | train_loss 1.3838 | time 15.6s
step 81400 | train_loss 1.4872 | time 15.6s
step 81600 | train_loss 1.5824 | time 15.6s
step 81800 | train_loss 1.4778 | time 15.6s
step 82000 | train_loss 1.4428 | time 15.6s
✅ step 82000 | val_loss 2.0636
💾 Saved LAST: /content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/LAST.pt
step 82200 | train_loss 1.5629 | time 18.3s
step 82400 | train_loss 1.3199 | time 15.6s
step 82600 | train_loss 1.4075 | time 15.6s
step 82800 | train_loss 1.4033 | time 15.6s
step 83000 | train_loss 1.4284 | time 15.6s
step 83200 | train_loss 1.4170 | time 15.6s
step 83400 | train_loss 1.6252 | time 15.6s
step 83600 | train_loss 1.4369 | time 15.6s
step 83800 | train_loss 1.5122 | time 15.6s
step 84000 | train_loss 1.5172 | time 15.6s
✅ st

KeyboardInterrupt: 

In [23]:
import torch

ckpt = torch.load(BEST_PATH, map_location=device)

if "model_state" in ckpt:
    model.load_state_dict(ckpt["model_state"], strict=True)
elif "model" in ckpt:
    model.load_state_dict(ckpt["model"], strict=True)
else:
    model.load_state_dict(ckpt, strict=True)

model.eval()

print("✅ BEST model loaded")
print("BEST step:", ckpt.get("step", "unknown"))
print("BEST val_loss:", ckpt.get("best_val", "unknown"))

✅ BEST model loaded
BEST step: 64000
BEST val_loss: 2.0251689106225967


In [26]:



import torch
import torch.nn.functional as F

@torch.no_grad()
def generate_text(prompt, model, tokenizer, max_new_tokens=80, temperature=0.45, top_k=15, eos_token_id=None, repetition_penalty=1.15):

    model.eval()

    enc = tokenizer.encode(prompt)
    input_ids = torch.tensor([enc.ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):

        outputs = model(input_ids)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]
        logits = logits[:, -1, :]

        # repetition penalty
        for token in set(input_ids[0].tolist()):
            logits[0, token] /= repetition_penalty

        logits = logits / temperature

        v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
        filtered = torch.full_like(logits, float("-inf"))
        filtered.scatter_(1, ix, v)
        logits = filtered

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=1)

        if eos_token_id is not None and next_token.item() == eos_token_id:
            break

    return tokenizer.decode(input_ids[0].tolist())
prompts = [
    "Once upon a time",
    "There was a little girl named Jenny",
    "Tim and Ben liked to play"
]

for p in prompts:
    print("\nPROMPT:", p)
    print(generate_text(
        p,
        model,
        tokenizer,
        max_new_tokens=80,
        temperature=0.45,
        top_k=15,
        eos_token_id=EOS_ID,
        repetition_penalty=1.15
    ))
    print("-" * 60)


PROMPT: Once upon a time
Once upon a time there a , was boy Tim He to with family One , parents and went the . had special for : new ! was big bright he . parents it so that couldn t him . they him it his to until got . was very . saw it and , he to . asked " is today His is birthday His replied His said " ' birthday The on ". parents , he ' not what say but knew
------------------------------------------------------------

PROMPT: There was a little girl named Jenny
There was a little girl named Jenny She three old . loved play her and new with friends One , went the with mom dad a for walk They to park Jenny s . they on way they a park Jenny a playground Jenny a with of . saw swings slides seesaws ran in asked " ' I to them Her said " !" parents and said Yes Jenny to so started climb the . Jenny climbing and she having much . when reached top she something
------------------------------------------------------------

PROMPT: Tim and Ben liked to play
Tim and Ben liked to play the . 

In [27]:
# Test the model with Computer Science questions

cs_prompts = [
"Explain what an array is in simple words.",
"What is a variable in programming?",
"Explain what a loop does in a computer program.",
"What is artificial intelligence?",
"Explain recursion in programming like I am a beginner."
]

for p in cs_prompts:
 print("\nPROMPT:", p)


 output = generate_text(
    p,
    model,
    tokenizer,
    max_new_tokens=80,
    temperature=0.45,
    top_k=15,
    eos_token_id=EOS_ID
)

print("AI ANSWER:", output)
print("-" * 70)



PROMPT: Explain what an array is in simple words.

PROMPT: What is a variable in programming?

PROMPT: Explain what a loop does in a computer program.

PROMPT: What is artificial intelligence?

PROMPT: Explain recursion in programming like I am a beginner.
AI ANSWER: Ex plain re cur sion in pro gram ming like I am a begin ner . day will be and . will to you your ent . ' sure , won t ." was and . did want be nar . said " I m ac , ac . can whatever want I ." started sell . made lot money He his with and . made of and things He his ry his . made people him . made unhappy angry sad confused He at . threw tantrum He and . made fuss he . made
----------------------------------------------------------------------


# Conclusion

In this project, I successfully built and trained a **GPT-style language model from scratch** using the TinyStories dataset. The model was able to learn basic narrative patterns and generate simple text continuations based on input prompts.

Although the generated text is not always grammatically perfect, the model demonstrates an understanding of story structure and token prediction. This outcome highlights both the potential and the limitations of training language models from scratch with limited computational resources.

This project provided valuable hands-on experience with:

* Transformer-based language model architecture
* Tokenization and vocabulary construction
* Dataset preparation for language modeling
* Training loops and checkpointing strategies
* Text generation techniques

Overall, this project strengthened my understanding of how modern language models work internally and the engineering required to train them. Future improvements could include larger model architectures, longer training time, and fine-tuning on domain-specific datasets to improve fluency and coherence.
